import torch
→ PyTorch library import karta hai.

import torch.nn as nn
→ Neural Network ke layers aur models banane ke liye.

import torch.optim as optim
→ Optimizers (Adam, SGD etc.) ke liye.

import torchvision
→ Computer Vision ke tools aur datasets ke liye.

import torchvision.transforms as transforms
→ Images par preprocessing/transformation apply karne ke liye.

from torch.utils.data import DataLoader
→ Dataset ko batches mein load karne ke liye.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [ ]:
transform = t]

In [5]:
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 501kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.8MB/s]


In [6]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

class MNISTClassifier(nn.Module):
→ MNIST ke liye Neural Network model ki class banata hai.

def __init__(self):
→ Model ke layers initialize karta hai.

super().__init__()
→ nn.Module ko initialize karta hai.

self.flatten = nn.Flatten()
→ Image ko 2D (28×28) se 1D (784) mein convert karta hai.

self.layers = nn.Sequential(...)
→ Multiple layers ko ek sequence mein arrange karta hai.

nn.Linear(784, 128)
→ 784 input values ko 128 neurons mein convert karta hai.

nn.ReLU()
→ Activation function apply karta hai.

nn.Linear(128, 10)
→ 128 neurons se 10 output classes (0–9) deta hai.

Forward Pass

def forward(self, x):
→ Input model ke andar kaise flow karega, define karta hai.

x = self.flatten(x)
→ Image ko 784 values mein flatten karta hai.

x = self.layers(x)
→ Image ko defined Linear → ReLU → Linear layers se pass karta hai.

return x
→ Final 10 class outputs return karta hai.

Overall

28×28 Image → Flatten (784) → Linear (128) → ReLU → Linear (10) → Prediction

👉 Ye simple Fully Connected Neural Network (ANN) hai jo MNIST digits 0–9 classify karta hai.

In [7]:
class MNISTClassifier(nn.Module) :
  def __init__(self) :
    super().__init__()
    self.flatten = nn.Flatten()
    self.layers = nn.Sequential(
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

  def forward(self, x) :
    x = self.flatten(x)
    x = self.layers(x)
    return x

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')

Using cpu


In [9]:
model = MNISTClassifier().to(device)

In [10]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

def train_epoch(model, train_loader, loss_function, optimizer, device):
→ 1 epoch ki training ke liye function banata hai.

model.train()
→ Model ko training mode mein set karta hai.

running_loss = 0.0
→ Loss ka total track karta hai.

correct = 0
→ Correct predictions ka count rakhta hai.

total = 0
→ Total samples ka count rakhta hai.

Training Loop

for batch_idx, (data, target) in enumerate(train_loader):
→ Training data ko batch-by-batch leta hai aur batch number bhi deta hai.

data, target = data.to(device), target.to(device)
→ Data aur labels ko GPU/CPU par bhejta hai.

optimizer.zero_grad()
→ Previous gradients ko clear karta hai.

output = model(data)
→ Model se prediction nikalta hai.

loss = loss_function(output, target)
→ Prediction aur actual target ke beech loss calculate karta hai.

loss.backward()
→ Loss ke basis par gradients calculate karta hai.

optimizer.step()
→ Gradients ke according weights update karta hai.

Loss & Accuracy

running_loss += loss.item()
→ Current batch ka loss add karta hai.

_, predicted = output.max(1)
→ Har image ke liye highest-score class ko prediction banata hai.

total += target.size(0)
→ Current batch ke total samples count karta hai.

correct += predicted.eq(target).sum().item()
→ Kitni predictions correct hain, count karta hai.

Every 100 Batches

if batch_idx % 100 == 0 and batch_idx > 0:
→ Har 100 batches ke baad progress print karta hai.

avg_loss = running_loss / 100
→ Last 100 batches ka average loss nikalta hai.

accuracy = 100. * correct / total
→ Training accuracy percentage calculate karta hai.

print(...)
→ Current batch, loss aur accuracy display karta hai.

running_loss = 0.0
→ Next 100 batches ke liye loss ko reset karta hai.

Overall Flow

Batch → Prediction → Loss → Backpropagation → Weight Update → Accuracy → हर 100 batches पर progress

👉 Ye function model ko 1 complete epoch ke liye train karta hai aur training progress dikhata hai.

In [11]:
def train_epoch(model, train_loader, loss_function, optimizer, device) :
  model.train()
  running_loss = 0.0
  correct = 0
  total = 0

  for batch_idx, (data, target) in enumerate(train_loader) :
    data, target = data.to(device), target.to(device)

    optimizer.zero_grad()
    output = model(data)
    loss = loss_function(output, target)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    _, predicted = output.max(1)
    total += target.size(0)
    correct += predicted.eq(target).sum().item()

    if batch_idx % 100 == 0 and batch_idx > 0 :
      avg_loss = running_loss / 100
      accuracy = 100. * correct / total
      print(f'[{batch_idx * 64}/{60000}]'
            f'Loss: {avg_loss:.3f} | Accuracy: {accuracy:.1f}%')
      running_loss = 0.0

def evaluate(model, test_loader, device):
→ Model ko test/evaluate karne ka function banata hai.

model.eval()
→ Model ko evaluation mode mein set karta hai.

correct = 0
→ Correct predictions ka count 0 se start karta hai.

total = 0
→ Total test samples ka count 0 se start karta hai.

with torch.no_grad():
→ Testing ke time gradient calculation off karta hai.

for inputs, targets in test_loader:
→ Test data ko batch-by-batch leta hai.

inputs, targets = inputs.to(device), targets.to(device)
→ Inputs aur labels ko GPU/CPU par bhejta hai.

outputs = model(inputs)
→ Model se predictions nikalta hai.

_, predicted = outputs.max(1)
→ Har image ke liye highest score wali class select karta hai.

total += targets.size(0)
→ Total samples ka count karta hai.

correct += predicted.eq(targets).sum().item()
→ Correct predictions ko count karta hai.

return 100. * correct / total
→ Test accuracy percentage return karta hai.

Overall

Test Data → Model → Prediction → Actual Label se Compare → Accuracy

👉 Ye function model ki test accuracy calculate karta hai.

In [15]:
def evaluate(model, test_loader, device) :
  model.eval()
  correct = 0
  total = 0

  with torch.no_grad() :
    for inputs, targets in test_loader :
      inputs, targets = inputs.to(device), targets.to(device)
      outputs = model(inputs)
      _, predicted = outputs.max(1)
      total += targets.size(0)
      correct += predicted.eq(targets).sum().item()

  return 100. * correct / total

num_epochs = 10
→ Model ko 10 epochs tak train karega.

for epoch in range(num_epochs):
→ Har epoch ke liye loop chalata hai.

print(f'\nEpoch : {epoch + 1}')
→ Current epoch number print karta hai.

train_epoch(model, train_loader, loss_function, optimizer, device)
→ Model ko 1 epoch train karta hai.

accuracy = evaluate(model, test_loader, device)
→ Trained model ki test accuracy calculate karta hai.

print(f'Test Accuracy : {accuracy:.2f}%')
→ Test accuracy ko 2 decimal places ke saath print karta hai.

Overall Flow

Epoch → Training → Testing → Accuracy → Next Epoch

👉 Ye loop 10 baar training + testing karta hai aur har epoch ke baad test accuracy dikhata hai.

In [16]:
num_epochs = 10
for epoch in range(num_epochs) :
  print(f'\nEpoch : {epoch + 1}')
  train_epoch(model, train_loader, loss_function, optimizer, device)
  accuracy = evaluate(model, test_loader, device)
  print(f'Test Accuracy : {accuracy:.2f}%')


Epoch : 1
[6400/60000]Loss: 0.086 | Accuracy: 97.8%
[12800/60000]Loss: 0.080 | Accuracy: 97.8%
[19200/60000]Loss: 0.084 | Accuracy: 97.8%
[25600/60000]Loss: 0.081 | Accuracy: 97.8%
[32000/60000]Loss: 0.090 | Accuracy: 97.7%
[38400/60000]Loss: 0.083 | Accuracy: 97.7%
[44800/60000]Loss: 0.087 | Accuracy: 97.7%
[51200/60000]Loss: 0.077 | Accuracy: 97.7%
[57600/60000]Loss: 0.075 | Accuracy: 97.7%
Test Accuracy : 97.13%

Epoch : 2
[6400/60000]Loss: 0.080 | Accuracy: 98.1%
[12800/60000]Loss: 0.073 | Accuracy: 98.1%
[19200/60000]Loss: 0.070 | Accuracy: 98.0%
[25600/60000]Loss: 0.076 | Accuracy: 98.0%
[32000/60000]Loss: 0.069 | Accuracy: 98.0%
[38400/60000]Loss: 0.081 | Accuracy: 97.9%
[44800/60000]Loss: 0.078 | Accuracy: 97.9%
[51200/60000]Loss: 0.082 | Accuracy: 97.9%
[57600/60000]Loss: 0.072 | Accuracy: 97.9%
Test Accuracy : 97.14%

Epoch : 3
[6400/60000]Loss: 0.073 | Accuracy: 98.0%
[12800/60000]Loss: 0.069 | Accuracy: 98.0%
[19200/60000]Loss: 0.071 | Accuracy: 98.0%
[25600/60000]Loss: 0.